# Fine-tune SmolLM2 for Medical Q&A (QLoRA)

Fine-tune **HuggingFaceTB/SmolLM2-360M-Instruct** on **medalpaca/medical_meadow_mediqa** with QLoRA — designed for Colab free tier (T4 GPU).

**Before you start**
1. Runtime → Change runtime type → **T4 GPU**
2. Run the setup cells below
3. Optional: mount Google Drive to persist checkpoints

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Set this to your GitHub username before running (skip if you opened the notebook from GitHub in Colab)
GITHUB_USER = "salisai"
REPO_NAME = "medical-slm"
REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"

# Clone only when src/ is missing (not needed if Colab opened the repo directly)
if not Path("src").exists():
    subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir(REPO_NAME)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
)

In [ ]:
import os
import sys
from pathlib import Path

# If running from notebooks/, add project root to Python path
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

# Optional: mount Google Drive for persistent checkpoints
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CHECKPOINT_DIR = "/content/drive/MyDrive/medical-slm/checkpoints"
else:
    CHECKPOINT_DIR = str(PROJECT_ROOT / "checkpoints")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoints will be saved to: {CHECKPOINT_DIR}")

In [ ]:
import torch
from huggingface_hub import login

from src.config import DatasetConfig, LoRAConfig, ModelConfig, TrainingConfig
from src.data_utils import load_medical_dataset
from src.inference import generate_response, load_model_for_inference
from src.train_utils import build_trainer

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Optional: login to push adapter to Hugging Face Hub
# login(token="hf_...")  # or set HF_TOKEN env var

## 1. Load dataset

The dataset has ~2,200 medical Q&A pairs in Alpaca format (`instruction`, `input`, `output`). We convert each row into chat messages for SmolLM2.

In [ ]:
dataset_config = DatasetConfig(
    dataset_name="medalpaca/medical_meadow_mediqa",
    max_samples=None,  # use full dataset (~2.2k rows)
)

train_dataset = load_medical_dataset(dataset_config)
print(f"Training samples: {len(train_dataset)}")
print(train_dataset[0])

## 2. Configure QLoRA and trainer

SmolLM2 is loaded in 4-bit. Only LoRA adapters on `q_proj` and `v_proj` are trained.

In [ ]:
model_config = ModelConfig(
    model_name="HuggingFaceTB/SmolLM2-360M-Instruct",
    bnb_4bit_compute_dtype="float16",
)

lora_config = LoRAConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
)

training_config = TrainingConfig(
    num_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    max_seq_length=512,
    fp16=True,
    gradient_checkpointing=True,
    save_steps=100,
    logging_steps=25,
    output_dir=CHECKPOINT_DIR,
    hub_model_id=None,  # e.g. "your-username/smol-medical-coach"
)

trainer = build_trainer(
    train_dataset=train_dataset,
    training_config=training_config,
    model_config=model_config,
    lora_config=lora_config,
)

print(f"Mixed precision: fp16={trainer.args.fp16}, bf16={trainer.args.bf16}")

## 3. Train and save

Expect ~20–40 minutes for 2 epochs on a Colab T4 GPU.

In [ ]:
trainer.train()
trainer.save_model(training_config.output_dir)

tokenizer = getattr(trainer, "processing_class", trainer.tokenizer)
tokenizer.save_pretrained(training_config.output_dir)
print(f"Adapter saved to {training_config.output_dir}")

## 4. Test the fine-tuned model

Load the saved LoRA adapter and ask a sample medical question.

In [ ]:
tokenizer, model = load_model_for_inference(
    model_name=model_config.model_name,
    adapter_path=training_config.output_dir,
)

test_question = (
    "I have been feeling very tired and dizzy lately. "
    "Could this be related to anemia? What should I ask my doctor?"
)

response = generate_response(tokenizer, model, test_question)
print("Question:", test_question)
print("\nAnswer:", response)